# Task : General Health Query Chatbot (Prompt Engineering Based)

## Objective
Create a chatbot that answers general health-related questions using an LLM with prompt engineering.

## Tools
- **Primary**: OpenAI GPT-3.5 Turbo via API
- **Free Alternative**: Mistral-7B-Instruct via Hugging Face Inference API

## Step 1: Install Libraries

In [ ]:
!pip install openai requests python-dotenv

## Step 2: Setup — API Key Configuration

In [ ]:
import os
import json
from datetime import datetime

# -----------------------------------------------
# SET YOUR API KEY HERE (or use environment variable)
# Option A: OpenAI
OPENAI_API_KEY = 'your-openai-api-key-here'

# Option B: Hugging Face (free)
HF_API_KEY = 'your-huggingface-api-token-here'
# Get free token at: https://huggingface.co/settings/tokens

# Choose which to use: 'openai' or 'huggingface'
MODEL_PROVIDER = 'openai'  # Change to 'huggingface' for free option
# -----------------------------------------------
print(f'Using provider: {MODEL_PROVIDER}')

## Step 3: Define the System Prompt (Prompt Engineering)

In [ ]:
# This is the core of prompt engineering — defining the chatbot's personality,
# scope, tone, and safety guidelines.

SYSTEM_PROMPT = """
You are a friendly and knowledgeable General Health Assistant.

Your role:
- Answer general health and wellness questions clearly and compassionately.
- Explain symptoms, common conditions, medications, and healthy habits in simple language.
- Encourage healthy lifestyle choices.

Important safety rules you MUST follow:
1. NEVER diagnose a medical condition. Always say 'This is general information, not a diagnosis.'
2. For serious symptoms (chest pain, difficulty breathing, stroke signs), ALWAYS advise the user to call emergency services immediately.
3. NEVER recommend specific prescription medications or dosages for serious conditions.
4. ALWAYS remind users to consult a licensed doctor or healthcare professional for personal medical advice.
5. If a question is outside general health (e.g., detailed surgery, rare diseases), politely acknowledge your limitations.
6. Maintain a warm, empathetic, and non-alarming tone.

Response format:
- Keep responses concise but informative (3-5 sentences for simple questions).
- Use bullet points for lists of symptoms or steps.
- End responses with a brief safety reminder when appropriate.
"""

print('System prompt defined!')
print(f'Prompt length: {len(SYSTEM_PROMPT)} characters')

## Step 4: Safety Filter Function

In [ ]:
# Pre-screening: detect queries that need immediate emergency guidance
EMERGENCY_KEYWORDS = [
    'chest pain', 'heart attack', 'stroke', 'cant breathe', "can't breathe",
    'difficulty breathing', 'unconscious', 'overdose', 'suicide', 'severe bleeding',
    'not breathing', 'choking'
]

OUT_OF_SCOPE_KEYWORDS = [
    'prescribe', 'surgery', 'operate', 'exact dosage for my', 'diagnose me with'
]

def safety_filter(user_message):
    """
    Pre-screens user input for emergency situations or out-of-scope requests.
    Returns (is_safe, response_override) — if not safe, return override message.
    """
    msg_lower = user_message.lower()
    
    # Check for emergency keywords
    for kw in EMERGENCY_KEYWORDS:
        if kw in msg_lower:
            return False, (
                '🚨 **This sounds like a medical emergency!**\n\n'
                'Please **call emergency services (115 / 1122 in Pakistan, or 911/999)** immediately.\n'
                'Do not wait — get professional medical help right away.'
            )
    
    # Check for out-of-scope requests
    for kw in OUT_OF_SCOPE_KEYWORDS:
        if kw in msg_lower:
            return False, (
                'I appreciate your trust, but I\'m not able to prescribe medications, '
                'provide diagnoses, or give surgical guidance. '
                'Please consult a licensed doctor for personalized medical advice. '
                'I can answer general health questions instead!'
            )
    
    return True, None


# Test the safety filter
test_cases = [
    'What causes a sore throat?',
    'I have severe chest pain right now',
    'Can you prescribe me antibiotics?'
]

print('Safety Filter Tests:')
for tc in test_cases:
    safe, override = safety_filter(tc)
    print(f'  "{tc}"')
    print(f'  → Safe: {safe}', '\n' if safe else f'  → Override: {override[:60]}...\n')

## Step 5: Build the Chatbot

In [ ]:
import requests

class HealthChatbot:
    def __init__(self, provider='openai', openai_key=None, hf_key=None):
        self.provider = provider
        self.openai_key = openai_key
        self.hf_key = hf_key
        self.conversation_history = []
        self.query_count = 0
    
    def ask(self, user_message):
        """Process a user health query and return a response."""
        self.query_count += 1
        print(f'\n{'='*55}')
        print(f'  User: {user_message}')
        print(f'{'='*55}')
        
        # Step 1: Run safety filter
        is_safe, override = safety_filter(user_message)
        if not is_safe:
            print(f'Bot: {override}')
            return override
        
        # Step 2: Add message to conversation history
        self.conversation_history.append({
            'role': 'user', 'content': user_message
        })
        
        # Step 3: Get response from LLM
        if self.provider == 'openai':
            response = self._call_openai()
        else:
            response = self._call_huggingface(user_message)
        
        # Step 4: Store response in history
        self.conversation_history.append({
            'role': 'assistant', 'content': response
        })
        
        print(f'Bot: {response}')
        return response
    
    def _call_openai(self):
        """Call OpenAI GPT-3.5 Turbo API."""
        import openai
        client = openai.OpenAI(api_key=self.openai_key)
        
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + self.conversation_history
        
        response = client.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=messages,
            max_tokens=500,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    
    def _call_huggingface(self, user_message):
        """Call Mistral-7B-Instruct via Hugging Face Inference API."""
        API_URL = 'https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.2'
        headers = {'Authorization': f'Bearer {self.hf_key}'}
        
        # Format prompt for Mistral instruction format
        prompt = f'[INST] {SYSTEM_PROMPT}\n\nUser query: {user_message} [/INST]'
        
        payload = {'inputs': prompt, 'parameters': {'max_new_tokens': 400, 'temperature': 0.7}}
        
        response = requests.post(API_URL, headers=headers, json=payload)
        result = response.json()
        
        if isinstance(result, list) and len(result) > 0:
            generated = result[0].get('generated_text', '')
            # Strip the prompt from the response
            if '[/INST]' in generated:
                return generated.split('[/INST]')[-1].strip()
            return generated.strip()
        return 'Sorry, I could not process your request. Please try again.'
    
    def reset_conversation(self):
        """Clear conversation history for a new session."""
        self.conversation_history = []
        print('Conversation history cleared.')


# Initialize the chatbot
chatbot = HealthChatbot(
    provider=MODEL_PROVIDER,
    openai_key=OPENAI_API_KEY,
    hf_key=HF_API_KEY
)
print('Health Chatbot initialized!')

## Step 6: Test the Chatbot with Example Queries

In [ ]:
# NOTE: These tests will only return real responses if you have set your API key above.
# If no key is provided, you'll see an authentication error — that's expected.
# Replace 'your-openai-api-key-here' with your actual key to see responses.

# Test Query 1: General symptom question
chatbot.ask('What causes a sore throat?')

In [ ]:
# Test Query 2: Medication safety question
chatbot.ask('Is paracetamol safe for children?')

In [ ]:
# Test Query 3: Lifestyle question
chatbot.ask('How much water should I drink daily?')

In [ ]:
# Test Query 4: Safety filter test — emergency keywords
chatbot.ask('I have severe chest pain and difficulty breathing')

In [ ]:
# Test Query 5: Out of scope test
chatbot.ask('Can you diagnose me with diabetes based on my symptoms?')

## Step 7: Interactive CLI Chatbot (Run in Terminal)

In [ ]:
def run_interactive_chatbot():
    """
    Interactive command-line chatbot session.
    Type 'quit' to exit, 'reset' to clear history.
    """
    print('\n' + '='*55)
    print('   🏥 General Health Query Chatbot')
    print('   DevelopersHub AI/ML Internship')
    print('='*55)
    print('Type your health question. Type "quit" to exit.\n')
    
    while True:
        try:
            user_input = input('You: ').strip()
            
            if not user_input:
                continue
            if user_input.lower() == 'quit':
                print('Thank you for using the Health Chatbot. Stay healthy!')
                break
            if user_input.lower() == 'reset':
                chatbot.reset_conversation()
                continue
            
            chatbot.ask(user_input)
            print()
        
        except KeyboardInterrupt:
            print('\nSession ended.')
            break

# Uncomment to run interactively:
# run_interactive_chatbot()

## Step 8: Prompt Engineering Analysis

In [ ]:
# Demonstrate different prompt engineering techniques used

prompt_techniques = {
    'Role Assignment': 'Act as a friendly health assistant — sets tone and persona',
    'Safety Rules': 'Explicit numbered rules prevent harmful outputs',
    'Response Format': 'Specifying bullet points and length controls output structure',
    'Scope Limitation': 'Defining what NOT to do prevents hallucinated advice',
    'Emergency Override': 'Safety filter runs BEFORE LLM call for critical keywords',
    'Conversation History': 'Multi-turn context allows follow-up questions'
}

print('Prompt Engineering Techniques Used:')
print('='*55)
for technique, explanation in prompt_techniques.items():
    print(f'\n✅ {technique}')
    print(f'   {explanation}')

## Step 9: Conclusion

- A **system prompt** was carefully engineered to define the bot's role, tone, scope, and safety constraints.
- A **pre-screening safety filter** handles emergency keywords before any LLM call.
- The bot supports **multi-turn conversation** by maintaining history context.
- Both **OpenAI GPT-3.5** (paid) and **Mistral-7B via Hugging Face** (free) are supported.
- **Key prompt engineering techniques**: role assignment, explicit safety rules, response format specification, and scope limitation.
- **Next steps**: Deploy with Streamlit UI, add voice input, or integrate a medical knowledge base.